# NL2FOL: public data loader and tiny SMT validity check

This notebook accompanies [Lalwani et al., arXiv:2405.02318](https://arxiv.org/abs/2405.02318). It loads the public logical-fallacy data used by the official implementation and demonstrates the paper's central solver test without an LLM or API key.

In [ ]:
%pip -q install z3-solver

In [ ]:
import csv
import io
import json
import urllib.request
from z3 import EnumSort, ForAll, Function, Implies, Not, Solver, BoolSort, sat

## 1. Load the public LOGIC-derived CSV

The official NL2FOL repository contains prepared general-domain and climate-domain files. This cell loads the general-domain fallacy file and shows only the fields needed for a compact example.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/lovishchopra/NL2FOL/main/data/fallacies.csv'
with urllib.request.urlopen(DATA_URL) as response:
    text = response.read().decode('utf-8-sig')
rows = list(csv.DictReader(io.StringIO(text)))

print(f'Loaded {len(rows)} examples.')
for row in rows[1:3]:
    compact = {key: row[key] for key in ('source_article', 'updated_label')}
    print(json.dumps(compact, indent=2, ensure_ascii=False))

## 2. Formalize an invalid generalization

Toy argument: *A tall person likes cheese; therefore every tall person likes cheese.* The premise describes one witness, while the conclusion quantifies over everyone. NL2FOL would generate the representation through language-model and NLI stages; here we write it explicitly so the symbolic step is easy to inspect.

In [ ]:
Person, (witness, other_person) = EnumSort('Person', ['witness', 'other_person'])
Tall = Function('Tall', Person, BoolSort())
LikesCheese = Function('LikesCheese', Person, BoolSort())
from z3 import Const
person = Const('person', Person)
premises = [Tall(witness), LikesCheese(witness)]
conclusion = ForAll([person], Implies(Tall(person), LikesCheese(person)))

solver = Solver()
solver.add(*premises, Not(conclusion))
status = solver.check()
print('premises AND NOT(conclusion):', status)
if status == sat:
    model = solver.model()
    print('Countermodel:')
    for individual in (witness, other_person):
        print(individual, {
            'Tall': bool(model.eval(Tall(individual), model_completion=True)),
            'LikesCheese': bool(model.eval(LikesCheese(individual), model_completion=True)),
        })
    print('Verdict: invalid / fallacious')

## 3. Contrast with a valid argument

Toy argument: *All cyclists wear helmets; Ana is a cyclist; therefore Ana wears a helmet.* No interpretation can satisfy the premises while making the conclusion false.

In [ ]:
Rider, (ana, ben) = EnumSort('Rider', ['ana', 'ben'])
Cyclist = Function('Cyclist', Rider, BoolSort())
WearsHelmet = Function('WearsHelmet', Rider, BoolSort())
rider = Const('rider', Rider)

premises = [
    ForAll([rider], Implies(Cyclist(rider), WearsHelmet(rider))),
    Cyclist(ana),
]
conclusion = WearsHelmet(ana)

solver = Solver()
solver.add(*premises, Not(conclusion))
status = solver.check()
print('premises AND NOT(conclusion):', status)
print('Verdict:', 'valid under this formalization' if status != sat else 'invalid')

### Scope

This reproduces the solver-side idea, not the full autoformalization system. Real performance depends heavily on whether entity extraction, predicates, quantifiers, and background knowledge accurately preserve the argument's meaning.